![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 03: Statistical Inference
***
**Learning objectives**
- Fit and interpret linear regression models for LOS and charges
- Assess regression assumptions (linearity, normality of residuals, homoscedasticity, independence)
- Apply transformations for skewed outcomes (log, Box-Cox)
- Interpret coefficients for continuous and categorical predictors
- Use model diagnostics plots: residual vs fitted, QQ plot, Cook's distance

**Estimated time:** 2 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `statsmodels`, `scipy`, `matplotlib`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import boxcox
import warnings
warnings.filterwarnings('ignore')
import os; os.makedirs('/tmp/mod03',exist_ok=True)

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

np.random.seed(42); N=1200
def logistic(x): return 1/(1+np.exp(-x))
base = np.random.normal(size=(N,3))
np.random.seed(99)
ages          = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes         = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers        = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
heart_failure = np.random.binomial(1,logistic(0.4*base[:,1]-1.2),N)
copd          = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
admit_type    = np.random.choice(['Emergency','Elective','Urgent'],N,p=[0.52,0.30,0.18])
n_comorb      = diabetes+heart_failure+copd

# LOS with realistic predictors (right-skewed)
los_true = (2.5 + 1.5*heart_failure + 0.8*diabetes + 0.5*copd
            + 0.05*ages + 0.6*(admit_type=='Emergency').astype(int)
            + np.random.exponential(1.5,N))
los_days = los_true.clip(1,30).round(0).astype(int)

# Total charge ~ LOS * daily rate (correlated)
charges = (los_days * np.random.uniform(1800,4200,N)
           + 2000*heart_failure + 1500*diabetes
           + np.random.normal(0,3000,N)).clip(500).round(2)

df = pd.DataFrame({
    'patient_id':[f'PT-{i:05d}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,'admit_type':admit_type,
    'diabetes':diabetes,'heart_failure':heart_failure,'copd':copd,'n_comorb':n_comorb,
    'los_days':los_days,'total_charge':charges,
})
df['age_std']  = (df['age']-df['age'].mean())/df['age'].std()
df['log_charge'] = np.log(df['total_charge'])
df['log_los']    = np.log(df['los_days'])

print(f"Dataset: {df.shape}")
df[['age','los_days','total_charge']].describe(percentiles=[.25,.50,.75]).round(1)


## 2. Simple Linear Regression — LOS ~ Age

In [ ]:
# Fit simple OLS
model_simple = smf.ols('los_days ~ age', data=df).fit()
print(model_simple.summary())


In [ ]:
# Plot regression line with CI
fig,axes = plt.subplots(1,2,figsize=(13,5))

# Panel A — scatter + regression line
ax = axes[0]
ax.scatter(df['age'],df['los_days'],alpha=0.15,s=15,color='#AEC6CF')
x_range = np.linspace(df['age'].min(),df['age'].max(),100)
pred = model_simple.get_prediction(pd.DataFrame({'age':x_range}))
mean_pred = pred.predicted_mean
ci_lower, ci_upper = pred.conf_int().T
ax.plot(x_range,mean_pred,color='#1F4E79',lw=2,label='OLS fit')
ax.fill_between(x_range,ci_lower,ci_upper,alpha=0.2,color='#1F4E79',label='95% CI')
ax.set_xlabel('Age (years)'); ax.set_ylabel('LOS (days)')
ax.set_title(f'LOS ~ Age  (R²={model_simple.rsquared:.3f})')
ax.legend(fontsize=9)
coef = model_simple.params['age']
ax.text(0.05,0.92,f'β = {coef:.3f} days per year (p = {model_simple.pvalues["age"]:.4f})',
        transform=ax.transAxes,fontsize=9,
        bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))

# Panel B — residuals
ax = axes[1]
residuals = model_simple.resid
ax.scatter(model_simple.fittedvalues,residuals,alpha=0.2,s=12,color='#4878CF')
ax.axhline(0,color='red',ls='--',lw=1.2)
ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted — simple model')

plt.tight_layout()
plt.savefig('/tmp/mod03/regression_simple.png',bbox_inches='tight'); plt.show()


## 3. Multiple Linear Regression — LOS ~ All Predictors

In [ ]:
# Multivariable model with clinical predictors
formula = ('los_days ~ age + C(sex) + C(payer, Treatment("Private")) '
           '+ diabetes + heart_failure + copd + C(admit_type, Treatment("Elective"))')
model_multi = smf.ols(formula, data=df).fit()
print(model_multi.summary())


In [ ]:
# Extract coefficients into a clean table
coef_df = pd.DataFrame({
    'Coefficient': model_multi.params,
    'Std Error'  : model_multi.bse,
    'CI_lo'      : model_multi.conf_int()[0],
    'CI_hi'      : model_multi.conf_int()[1],
    'p-value'    : model_multi.pvalues,
    'Significant': model_multi.pvalues < 0.05,
}).round(4)
print("\nCoefficient table (sorted by effect size):")
display(coef_df.drop('Intercept').sort_values('Coefficient',ascending=False))


## 4. Regression Assumption Diagnostics

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(13,10))
fig.suptitle('Regression diagnostics — LOS multivariable model', fontsize=13)

residuals = model_multi.resid
fitted    = model_multi.fittedvalues
influence = model_multi.get_influence()
cooks_d   = influence.cooks_distance[0]
std_resid = influence.resid_studentized_internal

# ── A: Residuals vs Fitted ────────────────────────────────────────────────────
ax = axes[0,0]
ax.scatter(fitted,residuals,alpha=0.2,s=12,color='#4878CF')
ax.axhline(0,color='red',ls='--',lw=1.2)
lowess_sm = sm.nonparametric.lowess(residuals,fitted,frac=0.3)
ax.plot(lowess_sm[:,0],lowess_sm[:,1],color='orange',lw=2,label='LOWESS trend')
ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals')
ax.set_title('A  Residuals vs Fitted\n(should be random scatter around 0)'); ax.legend(fontsize=8)

# ── B: QQ plot ────────────────────────────────────────────────────────────────
ax = axes[0,1]
(osm, osr), (slope,intercept,r) = stats.probplot(residuals, dist='norm')
ax.plot(osm,osr,'o',ms=3,alpha=0.4,color='#4878CF')
ax.plot(osm,slope*np.array(osm)+intercept,color='red',lw=1.5)
ax.set_xlabel('Theoretical quantiles'); ax.set_ylabel('Sample quantiles')
ax.set_title(f'B  QQ Plot of Residuals\n(R²={r**2:.3f}, should be ≈1)')

# ── C: Scale-Location (√|residuals| vs fitted) ────────────────────────────────
ax = axes[1,0]
sqrt_std_resid = np.sqrt(np.abs(std_resid))
ax.scatter(fitted,sqrt_std_resid,alpha=0.2,s=12,color='#6ACC65')
lowess_sl = sm.nonparametric.lowess(sqrt_std_resid,fitted,frac=0.3)
ax.plot(lowess_sl[:,0],lowess_sl[:,1],color='orange',lw=2)
ax.set_xlabel('Fitted values'); ax.set_ylabel('√|Standardised residuals|')
ax.set_title('C  Scale-Location (homoscedasticity)\n(should be a flat line)')

# ── D: Cook's Distance ────────────────────────────────────────────────────────
ax = axes[1,1]
ax.stem(range(len(cooks_d)),cooks_d,markerfmt=',',linefmt='#B0C4DE',basefmt='k-')
threshold = 4/len(df)
ax.axhline(threshold,color='red',ls='--',lw=1.2,label=f'Threshold (4/N = {threshold:.4f})')
n_inf = (cooks_d > threshold).sum()
ax.set_xlabel('Observation index'); ax.set_ylabel("Cook's distance")
ax.set_title(f"D  Cook's Distance — Influential Points\n({n_inf} observations above threshold)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/mod03/regression_diagnostics.png',bbox_inches='tight'); plt.show()


## 5. Log-Transformation for Skewed Outcomes

In [ ]:
# Total charge is right-skewed — log-transform before regression
print("Skewness check:")
print(f"  total_charge : {stats.skew(df.total_charge):.3f}")
print(f"  log_charge   : {stats.skew(df.log_charge):.3f}")
print()

formula_log = ('log_charge ~ age + C(sex) + C(payer, Treatment("Private")) '
               '+ diabetes + heart_failure + copd + los_days '
               '+ C(admit_type, Treatment("Elective"))')
model_log = smf.ols(formula_log, data=df).fit()

# Back-transform coefficients for interpretation
coef_log = pd.DataFrame({
    'β (log scale)'  : model_log.params,
    'exp(β)'         : np.exp(model_log.params),
    'p-value'        : model_log.pvalues,
}).round(4)
print("Log-charge model — back-transformed interpretation:")
display(coef_log.drop('Intercept').sort_values('exp(β)',ascending=False))
print()
print("Interpretation of exp(β):")
hf_expb = np.exp(model_log.params.get('heart_failure',0))
print(f"  Heart failure exp(β) = {hf_expb:.3f}")
print(f"  → Patients with HF have {(hf_expb-1)*100:.1f}% higher total charges on average")


## 6. Model Comparison with AIC/BIC

In [ ]:
# Compare models
results_comparison = pd.DataFrame({
    'Model'        : ['Simple (age only)','Multi (LOS outcome)','Log-charge model'],
    'R²'           : [model_simple.rsquared, model_multi.rsquared, model_log.rsquared],
    'Adj R²'       : [model_simple.rsquared_adj, model_multi.rsquared_adj, model_log.rsquared_adj],
    'AIC'          : [model_simple.aic, model_multi.aic, model_log.aic],
    'BIC'          : [model_simple.bic, model_multi.bic, model_log.bic],
    'N predictors' : [1, model_multi.df_model, model_log.df_model],
}).round(2)

print("Model comparison:")
display(results_comparison)
print()
print("Rules:")
print("  - Higher Adj R² = better fit, penalising extra predictors")
print("  - Lower AIC/BIC = better model (AIC: prediction; BIC: parsimony)")
print("  - AIC and BIC differences >10 = decisive evidence")


## 7. Knowledge Check
1. The coefficient for `heart_failure` in the LOS model is 1.82. Interpret this in one sentence.
2. The QQ plot tails deviate from the line. What does this suggest, and what should you do?
3. Cook's distance identifies 12 influential observations. Should you delete them?
4. Why is R² insufficient as the sole model selection criterion?
5. Re-fit the log-charge model adding `n_comorb` as a continuous predictor. Does AIC improve?


In [ ]:
# Exercise 5 — add n_comorb
formula_ex = ('log_charge ~ age + C(sex) + C(payer, Treatment("Private")) '
              '+ diabetes + heart_failure + copd + los_days + n_comorb '
              '+ C(admit_type, Treatment("Elective"))')
model_ex = smf.ols(formula_ex, data=df).fit()
print(f"Base model AIC : {model_log.aic:.2f}")
print(f"+ n_comorb AIC : {model_ex.aic:.2f}")
print(f"AIC improvement: {model_log.aic - model_ex.aic:.2f}")
print(f"n_comorb coef  : {model_ex.params.get('n_comorb',0):.4f}  (p={model_ex.pvalues.get('n_comorb',1):.4f})")


***
**Next → NB-05: Logistic Regression for Binary Outcomes**
